# 05 - Genie validation
Genie answers vs hand-written SQL (src/genie/validate.py), including the demo question whose answer changes after the Lakebase override.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
sp = ws().genie.get_space(os.environ.get('MI_GENIE_SPACE_ID', '01f1e2f665f81ba9aa29a999a763139c'))
print(sp.space_id, '|', sp.title, '| warehouse', sp.warehouse_id)

01f1e2f665f81ba9aa29a999a763139c | Matter Intelligence - Contract Portfolio | warehouse 8232b6cb7d06e103


In [3]:
sql("SELECT tag, run_id, count(*) questions, count_if(pass) passed, min(ts) ts FROM ${catalog}.ml.genie_validation GROUP BY ALL ORDER BY ts")

,tag,run_id,questions,passed,ts
0,dev-check,genie-dev-check-20261118-022101,7,7,2026-11-18T02:21:23.000Z
1,before,genie-before-20261118-031014,7,7,2026-11-18T03:10:36.000Z
2,after,genie-after-20261118-031339,1,1,2026-11-18T03:14:14.000Z


In [4]:
latest = sql("SELECT run_id FROM ${catalog}.ml.genie_validation WHERE tag = 'before' ORDER BY ts DESC LIMIT 1").iloc[0,0]
v = sql(f"SELECT q_no, question, pass, reason, genie_seconds, genie_sql FROM {CATALOG}.ml.genie_validation WHERE run_id = '{latest}' ORDER BY q_no")
v

,q_no,question,pass,reason,genie_seconds,genie_sql
0,1,Which contracts auto-renew in the next 90 days with uncapped liability?,true,same rows,20.7,"SELECT\n `title`,\n `contract_type`,\n `party_name`,\n `matter_team`,\n `next_renewal_date`,\n `non_renewal_no..."
1,2,Count of high-severity flags by contract type,true,same rows,12.9,"SELECT `d`.`contract_type`, COUNT(*) AS `high_severity_flags`\nFROM `serverless_stable_oj7eie_catalog`.`curated`.`go..."
2,3,What is the most common governing law?,true,same rows,13.8,"WITH governing_law_counts AS (\n SELECT\n `governing_law`,\n COUNT(*) AS `contracts`\n FROM `serverless_stab..."
3,4,Documents pending review by matter team,true,same rows,14.1,"SELECT `matter_team`, count(DISTINCT `doc_id`) AS `documents_pending_review`, count(*) AS `clauses_pending`\nFROM `s..."
4,5,"For each clause type that reviewers have touched, what share of decisions were overrides?",true,same rows,13.5,"SELECT\n `clause_type`,\n COUNT(*) AS `reviewer_decisions`,\n COUNT_IF(`review_action` = 'override') AS `override..."
5,6,How many contracts does each matter team own?,true,same rows,13.9,"SELECT `matter_team`, COUNT(*) AS `contracts_owned`\nFROM `serverless_stable_oj7eie_catalog`.`curated`.`gold_documen..."
6,7,List the auto-renewing contracts and their next renewal date,true,same rows,14.8,"SELECT `title`, `next_renewal_date`\nFROM `serverless_stable_oj7eie_catalog`.`curated`.`contract_terms_current`\nWHE..."


In [5]:
for r in v.itertuples(): print(f'Q{r.q_no}: {r.question}\n{r.genie_sql}\n')

Q1: Which contracts auto-renew in the next 90 days with uncapped liability?
SELECT
  `title`,
  `contract_type`,
  `party_name`,
  `matter_team`,
  `next_renewal_date`,
  `non_renewal_notice_deadline`,
  `notice_period_days`
FROM `serverless_stable_oj7eie_catalog`.`curated`.`contract_terms_current`
WHERE `auto_renewal` = true
  AND `uncapped_liability` = true
  AND `next_renewal_date` IS NOT NULL
  AND `title` IS NOT NULL
  AND `next_renewal_date` BETWEEN current_date() AND date_add(current_date(), 90)
ORDER BY `next_renewal_date` ASC, `title` ASC

Q2: Count of high-severity flags by contract type
SELECT `d`.`contract_type`, COUNT(*) AS `high_severity_flags`
FROM `serverless_stable_oj7eie_catalog`.`curated`.`gold_risk_flags` AS `f`
JOIN `serverless_stable_oj7eie_catalog`.`curated`.`gold_documents` AS `d`
  ON `f`.`doc_id` = `d`.`doc_id`
WHERE `f`.`severity` ILIKE '%high%'
  AND `d`.`contract_type` IS NOT NULL
GROUP BY `d`.`contract_type`
ORDER BY `high_severity_flags` DESC

Q3: What is

In [6]:
sql("SELECT tag, q_no, genie_rows, hand_rows, pass, ts FROM ${catalog}.ml.genie_validation WHERE q_no = 1 AND tag IN ('before','after') ORDER BY ts DESC LIMIT 2")

,tag,q_no,genie_rows,hand_rows,pass,ts
0,after,1,[],[],true,2026-11-18T03:14:14.000Z
1,before,1,"[[""JOINT CONTENT LICENSE AGREEMENT"", ""License"", ""WPT Enterprises, Inc.; ZYNGA INC.; ZYNGA GAME IRELAND LIMITED"", ""Te...","[[""JOINT CONTENT LICENSE AGREEMENT""]]",true,2026-11-18T03:10:36.000Z


In [7]:
sql("SELECT clause_type, reviewed_value, review_action, reviewer, review_comment, reviewed_at, synced_at FROM ${catalog}.curated.gold_reviewed_clauses WHERE clause_type = 'uncapped_liability'")

,clause_type,reviewed_value,review_action,reviewer,review_comment,reviewed_at,synced_at
0,uncapped_liability,false,override,erica.yuen@databricks.com,Carve-outs are subject to the aggregate cap per side letter; liability is capped,2026-11-18T03:12:42.537Z,2026-11-18T03:13:08.914Z
